# Forgetting Ledger — CIFAR experiments (Google Colab)

1. *Runtime → Change runtime type → GPU* (T4 is enough).
2. Add a Colab secret **`GH_TOKEN`** (key icon on the left) holding a GitHub token with read access to the private repository `Basil-Mohammad/forgetting-ledger`.
3. Run all cells. All outputs and checkpoints live on Google Drive, so after a disconnect simply **re-run all cells** — every job resumes from its last checkpoint.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/forgetting-ledger'
RUN_ROOT, DATA_ROOT, REPO_DIR = f'{BASE}/runs', '/content/data', '/content/forgetting-ledger'
import os; os.makedirs(RUN_ROOT, exist_ok=True)

In [ ]:
import os, subprocess
tok = userdata.get('GH_TOKEN')
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', f'https://{tok}@github.com/Basil-Mohammad/forgetting-ledger.git', REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
subprocess.run(['pip', '-q', 'install', '-r', f'{REPO_DIR}/requirements.txt'], check=True)
import torch; print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

In [ ]:
# Smoke test + timing on this GPU (about 1-2 minutes). Run once before the real jobs.
import subprocess, sys
subprocess.run([sys.executable, "scripts/run.py", "train", "configs/sc10_resnet.yaml", "seed=99", "n_tasks=2",
                "train_per_task=640", "train.epochs=1", f"out_root={RUN_ROOT}/_smoke", f"data_root={DATA_ROOT}"],
               cwd=REPO_DIR, check=True)

In [ ]:
# One line per job: <command> <config> [overrides...]
# `all` = train -> scores -> removal -> surgery -> lds -> params (each step resumable).
JOBS = """
all configs/sc10_resnet.yaml seed=0
all configs/sc10_resnet.yaml seed=1
all configs/sc10_resnet.yaml seed=2
all configs/sc10_resnet.yaml seed=3
all configs/sc10_resnet.yaml seed=4
train configs/sc10_resnet.yaml seed=0 learner.name=er
train configs/sc10_resnet.yaml seed=0 learner.name=derpp
train configs/sc10_resnet.yaml seed=0 learner.name=ewc
train configs/sc10_resnet.yaml seed=0 learner.name=agem
train configs/sc10_resnet.yaml seed=0 model.norm=bn
all configs/sc10_resnet.yaml seed=0 setting=task
"""
JOBS = [l.split() for l in JOBS.strip().splitlines() if l.strip() and not l.startswith("#")]
print(len(JOBS), "jobs")

In [ ]:
import subprocess, sys, time, os
def run_job(job):
    cmd = [sys.executable, "scripts/run.py", *job, f"out_root={RUN_ROOT}", f"data_root={DATA_ROOT}"]
    print(">>>", " ".join(job), flush=True)
    t0 = time.time()
    p = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    print(f"<<< exit {p.returncode} after {(time.time()-t0)/60:.1f} min", flush=True)
    return p.returncode

# Re-running this cell after a disconnect resumes exactly where it stopped:
# finished steps are skipped and the current step restarts from state/latest.pt.
for job in JOBS:
    if run_job(job) != 0:
        print("job failed - fix and re-run this cell (it will resume)"); break

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "flgr.analysis.aggregate", "--runs", RUN_ROOT, "--out", f"{RUN_ROOT}/../results"],
               cwd=REPO_DIR, check=True)
from IPython.display import Markdown, display
display(Markdown(open(f"{RUN_ROOT}/../results/results.md").read()))

In [ ]:
# Compact archive of everything needed for the paper (ledger tensors, scores, interventions, evals;
# model snapshots are excluded to keep it small).
import shutil, os, subprocess
out = f"{RUN_ROOT}/../flgr_results"
subprocess.run(f"cd {RUN_ROOT}/.. && zip -qr flgr_results.zip results runs -x '*/snapshots/*' '*/state/*'", shell=True)
print(os.path.getsize(f"{RUN_ROOT}/../flgr_results.zip")/1e6, "MB")